# Best Value Players — Contribution Per 90 Minutes
### Premier League 2025-26 | Until Gameday 31

---

## Project Question
**Which players deliver the most contribution relative to their playing time?**

Raw stats like goals and assists are biased toward players who play more minutes.
A player with 5 goals in 450 minutes is more impactful per game than one with
7 goals in 2500 minutes. This project normalizes all stats to a **per 90 minute**
basis, builds a custom **Value Score**, and ranks every player fairly.

---

## Steps
1. Load and clean data
2. Filter to players with meaningful minutes (avoid 5-min cameos)
3. Calculate per 90 stats for all key metrics
4. Build a custom Value Score per position
5. Rank and visualize the best value players
6. Conclusions

In [ ]:
# CELL 1 — Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

print('Libraries imported.')

In [ ]:
# CELL 2 — Load data
df = pd.read_csv('premier_league_complete_stats_until31thGameDayOnSeason2025-26.csv',
                 encoding='latin1')

print('Shape:', df.shape)
print('Teams:', df['team_name'].nunique())
print('Positions:', df['position'].unique())

In [ ]:
# CELL 3 — Filter to players with 450+ minutes
# Why 450? = 5 full 90-minute games
# Below this, per90 numbers get unreliable (one good game inflates everything)

MIN_MINUTES = 450
df_clean = df[df['minutesPlayed'] >= MIN_MINUTES].copy()

# Fill missing expectedGoals with 0 (86 nulls found earlier)
df_clean['expectedGoals'] = df_clean['expectedGoals'].fillna(0)
df_clean['expectedAssists'] = df_clean['expectedAssists'].fillna(0)

print(f'Players after filter: {len(df_clean)} (removed {len(df) - len(df_clean)} with < {MIN_MINUTES} mins)')
print(df_clean['position'].value_counts())

In [ ]:
# CELL 4 — Calculate Per 90 stats
# Per 90 formula: (stat / minutesPlayed) * 90
# This puts every player on equal footing regardless of how much they played

# List of stats to normalize
stats_to_normalize = [
    'goals', 'assists', 'expectedGoals', 'expectedAssists',
    'totalShots', 'shotsOnTarget', 'keyPasses',
    'successfulDribbles', 'tackles', 'interceptions',
    'bigChancesCreated', 'touches', 'wasFouled'
]

# Create new per90 column for each stat
for stat in stats_to_normalize:
    df_clean[stat + '_p90'] = (df_clean[stat] / df_clean['minutesPlayed']) * 90

print('Per90 columns created:')
print([col for col in df_clean.columns if '_p90' in col])

In [ ]:
# CELL 5 — Build Value Score per position
# Different positions contribute differently — a defender's value is not goals
# We weight stats differently based on what matters for each position
#
# MinMaxScaler scales each stat to 0-1 so they can be added fairly
# (without this, shots would dominate just because the numbers are bigger)

scaler = MinMaxScaler()

# --- FORWARDS: goals, shots, xG matter most ---
fwd_cols = ['goals_p90', 'assists_p90', 'expectedGoals_p90',
            'shotsOnTarget_p90', 'keyPasses_p90', 'successfulDribbles_p90']
fwd_weights = [0.30, 0.15, 0.20, 0.15, 0.10, 0.10]

# --- MIDFIELDERS: assists, key passes, dribbles, goals ---
mid_cols = ['goals_p90', 'assists_p90', 'keyPasses_p90',
            'bigChancesCreated_p90', 'successfulDribbles_p90', 'tackles_p90']
mid_weights = [0.20, 0.20, 0.20, 0.15, 0.15, 0.10]

# --- DEFENDERS: tackles, interceptions, plus any attacking contribution ---
def_cols = ['tackles_p90', 'interceptions_p90', 'goals_p90',
            'assists_p90', 'keyPasses_p90', 'wasFouled_p90']
def_weights = [0.30, 0.30, 0.10, 0.10, 0.10, 0.10]

def compute_value_score(subset, cols, weights):
    # Scale each column to 0-1 within the position group
    scaled = scaler.fit_transform(subset[cols])
    scaled_df = pd.DataFrame(scaled, columns=cols, index=subset.index)
    # Weighted sum = Value Score
    score = sum(scaled_df[col] * w for col, w in zip(cols, weights))
    return score

# Apply per position
fwd_df = df_clean[df_clean['position'] == 'F'].copy()
mid_df = df_clean[df_clean['position'] == 'M'].copy()
def_df = df_clean[df_clean['position'] == 'D'].copy()

fwd_df['value_score'] = compute_value_score(fwd_df, fwd_cols, fwd_weights)
mid_df['value_score'] = compute_value_score(mid_df, mid_cols, mid_weights)
def_df['value_score'] = compute_value_score(def_df, def_cols, def_weights)

# Combine back
df_scored = pd.concat([fwd_df, mid_df, def_df])

print('Value scores computed for', len(df_scored), 'players')

In [ ]:
# CELL 6 — Top 10 overall value players (table)

display_cols = ['player_name', 'team_name', 'position', 'minutesPlayed',
                'goals', 'assists', 'goals_p90', 'assists_p90', 'value_score']

top10 = (df_scored
         .sort_values('value_score', ascending=False)
         .head(10)[display_cols]
         .reset_index(drop=True))

top10.index += 1
top10.columns = ['Player', 'Team', 'Pos', 'Mins',
                 'Goals', 'Assists', 'Goals/90', 'Assists/90', 'Value Score']
top10.round(3)

In [ ]:
# CELL 7 — PLOT 1: Top 10 Value Players — Horizontal Bar
# Color-coded by position so you can see which positions dominate

top10_plot = df_scored.sort_values('value_score', ascending=True).tail(10)

pos_colors = {'F': 'tomato', 'M': 'steelblue', 'D': 'mediumseagreen'}
bar_colors = [pos_colors[p] for p in top10_plot['position']]

plt.figure(figsize=(11, 7))
bars = plt.barh(top10_plot['player_name'], top10_plot['value_score'],
                color=bar_colors, edgecolor='white')

# Value label at end of each bar
for i, v in enumerate(top10_plot['value_score']):
    plt.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=10)

# Manual legend for position colors
legend_patches = [
    mpatches.Patch(color='tomato',         label='Forward'),
    mpatches.Patch(color='steelblue',      label='Midfielder'),
    mpatches.Patch(color='mediumseagreen', label='Defender')
]
plt.legend(handles=legend_patches, loc='lower right')

plt.title('Top 10 Best Value Players — Per 90 Min Value Score (PL 2025-26)')
plt.xlabel('Value Score (0 to 1)')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 8 — PLOT 2: Top 10 per position (3 subplots)
# subplot(1,3,n) = 1 row, 3 columns, nth chart
# Each position ranked independently by their own value score

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

position_data = [
    (fwd_df, 'Forward',    'tomato'),
    (mid_df, 'Midfielder', 'steelblue'),
    (def_df, 'Defender',   'mediumseagreen')
]

for ax, (pos_df, label, color) in zip(axes, position_data):
    top = pos_df.sort_values('value_score', ascending=True).tail(10)
    ax.barh(top['player_name'], top['value_score'],
            color=color, edgecolor='white')
    ax.set_title(f'Top 10 {label}s')
    ax.set_xlabel('Value Score')
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('Best Value Players by Position — Per 90 Min Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 9 — PLOT 3: Scatter — Goals/90 vs Assists/90
# Bubble size = value score (bigger = better overall value)
# Top-right = players who both score AND create at high rates per 90

df_att = df_scored[df_scored['position'].isin(['F', 'M'])].copy()

plt.figure(figsize=(13, 8))

pos_colors = {'F': 'tomato', 'M': 'steelblue'}

for pos, group in df_att.groupby('position'):
    plt.scatter(
        group['goals_p90'],
        group['assists_p90'],
        s=group['value_score'] * 800,   # scale bubble size to value score
        color=pos_colors[pos],
        alpha=0.65,
        edgecolors='white',
        linewidth=0.5,
        label='Forward' if pos == 'F' else 'Midfielder'
    )

# Label top 10 by value score
for _, row in df_att.nlargest(10, 'value_score').iterrows():
    plt.annotate(row['player_name'],
                 (row['goals_p90'], row['assists_p90']),
                 textcoords='offset points',
                 xytext=(6, 4), fontsize=8)

plt.title('Goals per 90 vs Assists per 90 — Bubble Size = Value Score')
plt.xlabel('Goals per 90 mins')
plt.ylabel('Assists per 90 mins')
plt.legend()
plt.figtext(0.92, 0.02, 'Bigger bubble = higher value score',
            ha='right', fontsize=9, color='gray')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 10 — PLOT 4: Raw goals vs Goals per 90 comparison
# This is the core argument of the project:
# Raw stats lie — per90 reveals true efficiency
# Players who rank high in raw goals but low in per90 = played lots of minutes

forwards = fwd_df.sort_values('goals', ascending=False).head(15)

x = np.arange(len(forwards))
width = 0.35

fig, ax1 = plt.subplots(figsize=(14, 6))

# Left axis: raw goals
bars1 = ax1.bar(x - width/2, forwards['goals'],
                width, label='Raw Goals', color='tomato', alpha=0.85)
ax1.set_ylabel('Raw Goals', color='tomato')
ax1.tick_params(axis='y', labelcolor='tomato')

# Right axis: goals per 90 (separate scale)
ax2 = ax1.twinx()   # twinx() creates a second y-axis sharing the same x-axis
bars2 = ax2.bar(x + width/2, forwards['goals_p90'].round(2),
                width, label='Goals per 90', color='steelblue', alpha=0.85)
ax2.set_ylabel('Goals per 90', color='steelblue')
ax2.tick_params(axis='y', labelcolor='steelblue')

ax1.set_xticks(x)
ax1.set_xticklabels(forwards['player_name'], rotation=40, ha='right', fontsize=9)
ax1.set_title('Raw Goals vs Goals per 90 — Top 15 Forwards\n(Same player, different perspective)')

# Combined legend from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# CELL 11 — PLOT 5: Heatmap — Per90 Stats for Top 15 Value Players
# Rows = players, Columns = per90 stats
# Shows the full profile of each top-value player at once

top15 = df_scored.sort_values('value_score', ascending=False).head(15)

heatmap_cols = ['goals_p90', 'assists_p90', 'expectedGoals_p90',
                'shotsOnTarget_p90', 'keyPasses_p90',
                'successfulDribbles_p90', 'tackles_p90']

heatmap_data = top15.set_index('player_name')[heatmap_cols]

# Rename columns for cleaner labels
heatmap_data.columns = ['Goals/90', 'Assists/90', 'xG/90',
                        'SoT/90', 'KeyPass/90', 'Dribbles/90', 'Tackles/90']

plt.figure(figsize=(13, 8))

sns.heatmap(heatmap_data, annot=True, fmt='.2f',
            cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Stat value per 90 mins'})

plt.title('Per 90 Min Profile — Top 15 Value Players')
plt.xlabel('Stat (per 90 mins)')
plt.ylabel('Player')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 12 — PLOT 6: Value Score Distribution by Position
# Violin plot: shows how value scores are spread within each position
# Are forwards more consistently valuable, or is there more variance?

pos_map = {'F': 'Forward', 'M': 'Midfielder', 'D': 'Defender'}
df_scored['position_full'] = df_scored['position'].map(pos_map)

plt.figure(figsize=(10, 6))

sns.violinplot(data=df_scored, x='position_full', y='value_score',
               palette={'Forward': 'tomato', 'Midfielder': 'steelblue',
                        'Defender': 'mediumseagreen'},
               order=['Forward', 'Midfielder', 'Defender'],
               inner='quartile')

plt.title('Value Score Distribution by Position')
plt.xlabel('Position')
plt.ylabel('Value Score')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 13 — PLOT 7: Team Average Value Score
# Which teams have the most valuable players on average?
# groupby team, take mean of value_score, sort and plot

team_value = (df_scored
              .groupby('team_name')['value_score']
              .mean()
              .sort_values(ascending=True))

plt.figure(figsize=(11, 8))

bars = plt.barh(team_value.index, team_value.values,
                color=sns.color_palette('coolwarm', len(team_value)),
                edgecolor='white')

for i, v in enumerate(team_value.values):
    plt.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)

plt.title('Average Player Value Score by Team\n(Higher = more efficient squad per 90 mins)')
plt.xlabel('Average Value Score')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 14 — PLOT 8: Minutes Played vs Value Score
# Key insight check: are high-value players being played enough?
# If top-value players have low minutes = underused talent

plt.figure(figsize=(12, 7))

pos_colors = {'F': 'tomato', 'M': 'steelblue', 'D': 'mediumseagreen'}
pos_labels = {'F': 'Forward', 'M': 'Midfielder', 'D': 'Defender'}

for pos, group in df_scored.groupby('position'):
    plt.scatter(group['minutesPlayed'], group['value_score'],
                color=pos_colors[pos], alpha=0.6,
                label=pos_labels[pos], s=50,
                edgecolors='white', linewidth=0.3)

# Annotate top 8 value players
for _, row in df_scored.nlargest(8, 'value_score').iterrows():
    plt.annotate(row['player_name'],
                 (row['minutesPlayed'], row['value_score']),
                 textcoords='offset points',
                 xytext=(5, 4), fontsize=8)

plt.title('Minutes Played vs Value Score — Are High-Value Players Being Used?')
plt.xlabel('Minutes Played')
plt.ylabel('Value Score')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# CELL 15 — Final Rankings Table
# Clean ranked output — what you'd put in a README or report

final_table = (df_scored
               .sort_values('value_score', ascending=False)
               .head(20)[['player_name', 'team_name', 'position',
                           'minutesPlayed', 'goals', 'assists',
                           'goals_p90', 'assists_p90', 'value_score']]
               .reset_index(drop=True))

final_table.index += 1
final_table.columns = ['Player', 'Team', 'Pos', 'Mins',
                       'Goals', 'Assists', 'G/90', 'A/90', 'Value Score']
final_table.round(3)

---
## Conclusions

**What we found:**
- Raw stats (total goals/assists) favour players who simply play more minutes
- Per 90 normalization reveals which players are genuinely efficient
- The Value Score weights stats differently per position — a defender is not judged on goals
- Some high-value players have low total minutes — potentially underused by their team

**Key insight:**
A player with 8 goals in 700 minutes (1.03 goals/90) delivers more
attacking value than a player with 12 goals in 2500 minutes (0.43 goals/90),
even though the raw total looks better.

**Limitations:**
- Value Score weights are manually chosen — could be improved with regression
- Doesn't account for team quality or opposition faced
- Goalkeepers excluded (different stats entirely)

**Next steps:**
- Use linear regression to find which stats actually predict match outcomes
- Compare value scores across multiple seasons